In [1]:
import glob
import os
import datetime
from astropy.table import Table
filtername = 'F480M'
basepath = '/orange/adamginsburg/jwst/w51/'



In [2]:
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np

nircam_short_filters = ['F140M', 'F162M', 'F182M', 'F187N', 'F210M' ]
nircam_long_filters = [ 'F335M',  'F360M','F405N', 'F410M',  'F480M']
miri_filters = ['F560W', 'F770W', 'F1000W',  'F1280W',  'F2100W', ]
def tblname_to_imgname(tblname,  proposal_id='6151'):
    filtername = tblname.split('/')[-1].split('_')[0]
    if filtername.upper() in miri_filters:
        target= 'w51_miri'
    else:        
        target = 'w51'

    nvisits = {'2221': {'brick': 1, 'cloudc': 2},
                '1182': {'brick': 2},
                '6151': {'w51': 1, 'w51_miri': 2}
                }
    field_to_reg_mapping = {'2221': {'001': 'brick', '002': 'cloudc'},
                            '1182': {'004': 'brick'},
                            '6151': {'001': 'w51', '002':'w51_miri'}}[proposal_id]
    reg_to_field_mapping = {v:k for k,v in field_to_reg_mapping.items()}
    field = reg_to_field_mapping[target]


    
    module = tblname.split('/')[-1].split('_')[1]
    visit = tblname.split('/')[-1].split('_')[2]
    visitid = visit[5:]
    vgroup = tblname.split('/')[-1].split('_')[3]
    vgroupid = vgroup[6:]
    exposure = tblname.split('/')[-1].split('_')[4]
    expid = exposure[3:]
    imgname = f'{basepath}/{filtername.upper()}/pipeline/jw0{proposal_id}{field}{visitid}_{vgroupid}_{expid}_{module}_cal.fits'


    return imgname
def load_data(filename):
    fh = fits.open(filename)
    im1 = fh
    data = im1['SCI'].data
    try:
        wht = im1['WHT'].data
    except KeyError:
        wht = None
    err = im1['ERR'].data
    instrument = im1[0].header['INSTRUME']
    telescope = im1[0].header['TELESCOP']
    obsdate = im1[0].header['DATE-OBS']
    return fh, im1, data, wht, err, instrument, telescope, obsdate
def filter_by_nmatch(tbls_merged, tbls, tolerance=0):

    for jj, tbl_merged in enumerate(tbls_merged):
        # convert skycoord of tbl_merged to pixel coordinates in each tbl in tbls
        skycoord_merged = tbl_merged['skycoord']
        for ii, tbl in enumerate(tbls):
            img_name = tblname_to_imgname(tblfns[ii])
            fh, im1, img_data, wht, err, instrument, telescope, obsdate = load_data(img_name)
            wcs = WCS(im1[1].header)
            pixcoord_merged = wcs.world_to_pixel(skycoord_merged)
            
            
            # check whether pixel coordiantes of tbl_merged fall within the image field of view
         
            
            img_shape = img_data.shape
            in_fov = (pixcoord_merged[0] >= 0) & (pixcoord_merged[0] < img_shape[1]) & (pixcoord_merged[1] >= 0) & (pixcoord_merged[1] < img_shape[0])
            in_fov_int = in_fov.astype(int)
            
            # stack in_fov while looping over tbls to get a final mask of which sources in tbl_merged are in the field of view of at least one tbl in tbls
            if ii == 0:
                in_fov_all = in_fov_int
            else:
                in_fov_all = in_fov_all + in_fov_int

        nmatch_max = in_fov_all
        nmatch = tbl_merged['nmatch']
        from_sat_cat = tbl_merged['from_sat_catalog']
        from_sat_cat = from_sat_cat.astype(bool)
        keep = (nmatch >= nmatch_max - tolerance) | from_sat_cat
        tbl_merged_cut = tbl_merged[keep]
        print(f"Number of sources in merged table: {len(tbl_merged)}")
        print(f"Number of sources in merged table with nmatch >= {nmatch_max}: {len(tbl_merged_cut)}")  
        print(f"Number of sources in merged table with nmatch >= {nmatch_max - tolerance}: {len(tbl_merged[nmatch >= nmatch_max - tolerance])}")    
        #save the cut table to a new file
        if tolerance == 0:
            label = 'grade_a'
        elif tolerance ==1:
            label = 'grade_b'
        tbl_merged_cut.write(tblfns_merged[jj].replace('.fits', f'_nmatch_cut_{label}.fits'), overwrite=True)
    
            


In [ ]:
filternames=  miri_filters
for filtername in filternames:
    tblfns = glob.glob(f'{basepath}/{filtername.upper()}/*daophot_combined_with_satstars.fits')
    for tbl in tblfns:
        print(tbl)
        print('last modified:', datetime.datetime.fromtimestamp(os.path.getmtime(tbl)))

    tblfns_merged = glob.glob(f'/orange/adamginsburg/jwst/w51/catalogs/{filtername.lower()}_*_indivexp_merged_dao_after_merger_combined_with_satstars.fits')
    print('last modified:', datetime.datetime.fromtimestamp(os.path.getmtime(tblfns_merged[0])))

    tbls = [Table.read(tblfn) for tblfn in tblfns]
    tbls_merged = [Table.read(tblfn) for tblfn in tblfns_merged]


    filter_by_nmatch(tbls_merged, tbls, tolerance=0)
    filter_by_nmatch(tbls_merged, tbls, tolerance=1)

/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit002_vgroup0210b_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:29
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit001_vgroup02101_exp00004_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:27
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit001_vgroup02101_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:26
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit001_vgroup0210b_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:27
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit002_vgroup02101_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:30
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit001_vgroup02101_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:25
/orange/adamginsburg/jwst/w51//F560W/f560w_mirimage_visit002_vgroup02101_exp00001_

Set DATE-AVG to '2024-09-08T12:32:25.589' from MJD-AVG.
Set DATE-END to '2024-09-08T12:32:32.527' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.702250 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291148460.660 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-09-08T10:15:23.005' from MJD-AVG.
Set DATE-END to '2024-09-08T10:15:29.943' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.806250 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291912377.206 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/astropy/wcs/wcsapi/fitswcs.py:367: UserWarning: 'WCS.all_world2pix' failed to converge to the requested accuracy.
After 20 iterations, the solution is diverging at least for one input point.
  warnings.warn(str(e))
Set DATE-AVG to '2024-09-08T10:13:45.853' from MJD-AVG.
Set DATE-END to '2024-09-08T10:13:52.791' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.807478 from OBSGEO-[XYZ].
Set OBSGEO-H to 129192142

Number of sources in merged table: 55389
Number of sources in merged table with nmatch >= [6 6 6 ... 6 6 6]: 3327
Number of sources in merged table with nmatch >= [6 6 6 ... 6 6 6]: 3318
Number of sources in merged table: 55389
Number of sources in merged table with nmatch >= [6 6 6 ... 6 6 6]: 7255
Number of sources in merged table with nmatch >= [5 5 5 ... 5 5 5]: 7247
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit002_vgroup02103_exp00001_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:33
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit002_vgroup0210d_exp00002_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:34
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit001_vgroup0210d_exp00004_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:31
/orange/adamginsburg/jwst/w51//F770W/f770w_mirimage_visit002_vgroup0210d_exp00003_daophot_combined_with_satstars.fits
last modified: 2026-03-12 16:04:34
/orange/adamgi

Set DATE-AVG to '2024-09-08T11:54:02.295' from MJD-AVG.
Set DATE-END to '2024-09-08T11:54:09.233' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    -3.731399 from OBSGEO-[XYZ].
Set OBSGEO-H to 1291362083.380 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
